# Pre-modeling simple: base PNB + base VAR
### Econometría II
#### Clase introductoria a Google Colab

**Objetivo:** limpiar dos bases trimestrales y armar una tabla con **una fila por año**.

1. Explorar y limpiar (nulos, duplicados, errores, imputar)
2. Pasar a anual y unir las dos bases
3. Crear variables nuevas

Ejecuta cada celda con **Shift + Enter**, en orden.

## 0. Preparación

**Librerías** = herramientas ya programadas que cargamos con `import`:
- `pandas` (`pd`): para trabajar con **tablas**, como Excel.
- `numpy` (`np`): para cálculos. Aquí solo usamos `np.nan`, que significa **"dato vacío"**.
- `matplotlib` (`plt`): para **gráficos**.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

**Cargar los archivos.** Primero arrastra los dos Excel al panel **📁 Archivos** de la izquierda. Luego ejecuta la celda.

`pd.read_excel("nombre.xlsx")` lee el Excel y lo guarda como tabla. El nombre debe ser **exactamente** igual al del panel.

In [2]:
pnb = pd.read_excel("base_PNB - ficticia.xlsx")
var = pd.read_excel("base_VAR_IPX_IPM_Inversion - ficticia - copia.xlsx")

FileNotFoundError: [Errno 2] No such file or directory: 'base_PNB - ficticia.xlsx'

---
## 1. Explorar y limpiar

### 1.1 Ver los datos
`.head()` muestra las primeras 5 filas. Cada fila es **un trimestre**.

In [ ]:
pnb.head()

In [ ]:
pnb.head(10)

In [ ]:
var.head()

### 1.2 Tipos de dato
`.info()` muestra cada columna y su tipo. Debería ser `float64` (número). Si dice `object` (o `str`), la columna tiene **texto**. `Periodo` es texto a propósito (`2004Q1`).

In [ ]:
pnb.info()

In [ ]:
var.info()

`Gasto_pub` y `Ahorro` aparecen como `object`: tienen celdas con el texto `"N.A"` o `"n.a"`.

`pd.to_numeric(..., errors="coerce")` convierte la columna a número. Lo que es texto se vuelve **vacío** (`NaN`).

In [ ]:
pnb["Gasto_pub"] = pd.to_numeric(pnb["Gasto_pub"], errors="coerce")
var["Ahorro"] = pd.to_numeric(var["Ahorro"], errors="coerce")

### 1.3 Valores vacíos (nulos)
`.isna().sum()` cuenta cuántos vacíos hay en cada columna.

In [ ]:
pnb.isna().sum()

In [ ]:
var.isna().sum()

### 1.4 Duplicados
- `.duplicated().sum()` cuenta **filas repetidas**.
- `["Periodo"].duplicated().sum()` cuenta **trimestres repetidos**.

In [ ]:
print("Filas repetidas PNB:", pnb.duplicated().sum())
print("Filas repetidas VAR:", var.duplicated().sum())
print("Trimestres repetidos PNB:", pnb["Periodo"].duplicated().sum())
print("Trimestres repetidos VAR:", var["Periodo"].duplicated().sum())

No hay filas ni trimestres repetidos.

Pero hay otro problema: **valores copiados**. Que el PBI de dos trimestres distintos sea exactamente igual (con todos sus decimales) es casi imposible; seguro es un error de copiar y pegar.

`pnb["PBI"].duplicated()` marca los valores que ya aparecieron antes en la columna.

In [ ]:
pnb[pnb["PBI"].duplicated()]

2004Q4 tiene el mismo PBI que 2004Q2. Revisamos las cuatro variables de flujo y convertimos las copias en vacío.

Cómo leer la línea `pnb.loc[pnb["PBI"].duplicated(), "PBI"] = np.nan`:
- `pnb.loc[filas, columna]` = "en estas filas y esta columna..."
- `pnb["PBI"].duplicated()` = las filas donde el valor está repetido
- `= np.nan` = "...pon vacío"

In [ ]:
pnb.loc[pnb["PBI"].duplicated(), "PBI"] = np.nan
pnb.loc[pnb["Gasto_pub"].duplicated(), "Gasto_pub"] = np.nan
var.loc[var["Inversion"].duplicated(), "Inversion"] = np.nan
var.loc[var["Ahorro"].duplicated(), "Ahorro"] = np.nan

No lo hacemos con IPX e IPM: son índices con solo 2 decimales, así que dos trimestres con el mismo valor sí es posible.

### 1.5 Valores imposibles
Graficamos el gasto público para ver si hay algo raro.

In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(pnb["Periodo"], pnb["Gasto_pub"], marker="o", markersize=3)
plt.xticks(pnb["Periodo"][::8])        # mostrar una etiqueta cada 8 trimestres
plt.title("Gasto público")
plt.show()

Se ven dos caídas imposibles:
- **2023Q3**: el gasto vale **0**.
- **2008Q2**: el gasto vale **5,569**, cuando sus vecinos están cerca de **65,000** (diez veces menos).

Los convertimos en vacío.

In [ ]:
pnb.loc[pnb["Gasto_pub"] <= 0, "Gasto_pub"] = np.nan              # el cero
pnb.loc[pnb["Periodo"] == "2008Q2", "Gasto_pub"] = np.nan         # el valor diez veces menor

### 1.6 Saltos grandes en el PBI

In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(pnb["Periodo"], pnb["PBI"], marker="o", markersize=3, color="darkred")
plt.xticks(pnb["Periodo"][::8])
plt.title("PBI trimestral")
plt.show()

El PBI cae a la mitad en **2006** y en **2020**, y no se recupera. Parece un **cambio de escala o de año base**, no una crisis. **No lo corregimos** (hay que revisarlo en la fuente), pero lo marcaremos en el paso 3.

### 1.7 Imputar (rellenar los vacíos)
En series de tiempo, el mejor estimado de un dato vacío son **sus vecinos**: el trimestre anterior y el siguiente.

`.interpolate()` rellena cada vacío con una línea recta entre el dato de antes y el de después. `limit_direction="both"` también rellena el primer y el último trimestre si están vacíos.

In [ ]:
pnb[["PBI", "Gasto_pub"]] = pnb[["PBI", "Gasto_pub"]].interpolate(limit_direction="both")
var[["IPX", "IPM", "Inversion", "Ahorro"]] = var[["IPX", "IPM", "Inversion", "Ahorro"]].interpolate(limit_direction="both")

Comprobamos que ya no queden vacíos:

In [ ]:
print(pnb.isna().sum().sum(), "vacíos en PNB")
print(var.isna().sum().sum(), "vacíos en VAR")

---
## 2. Pasar a anual y unir

### 2.1 De trimestral a anual
`groupby("Anio")` junta los 4 trimestres de cada año. Luego decimos qué hacer con cada variable:
- **PBI, Gasto, Inversión, Ahorro** son **flujos**: el total del año es la **suma** (`"sum"`).
- **IPX, IPM** son **índices de precios**: se usa el **promedio** (`"mean"`). Sumarlos no tendría sentido.

`.reset_index()` deja `Anio` como una columna normal.

In [ ]:
pnb_anual = pnb.groupby("Anio").agg({"PBI": "sum", "Gasto_pub": "sum"}).reset_index()
pnb_anual.head()

In [ ]:
var_anual = var.groupby("Anio").agg({"IPX": "mean", "IPM": "mean",
                                     "Inversion": "sum", "Ahorro": "sum"}).reset_index()
var_anual.head()

### 2.2 Unir las dos bases
`merge` junta dos tablas usando una columna en común (`on="Anio"`).

`how="left"` conserva todos los años de la tabla de la izquierda (PNB, 2004–2024). El año 2003, que solo está en VAR, queda fuera porque no tiene PBI.

In [ ]:
tabla = pnb_anual.merge(var_anual, on="Anio", how="left")
tabla

✅ Una fila por año: 21 filas (2004 a 2024).

In [ ]:
print("Filas:", len(tabla))
print("¿Años repetidos?", tabla["Anio"].duplicated().sum())

---
## 3. Crear variables

| Variable | Fórmula | Qué mide |
|---|---|---|
| `TI` | IPX / IPM × 100 | términos de intercambio |
| `g_PBI`, `g_Inversion`, ... | variación % anual | crecimiento |
| `inv_pbi`, `gasto_pbi` | X / PBI × 100 | peso en la economía |
| `brecha` | (Ahorro − Inversión) / PBI × 100 | si es negativa, falta ahorro interno |
| `g_PBI_anterior` | crecimiento del año pasado | inercia |
| `quiebre` | 1 en años con el problema de escala | marcar años raros |

### 3.1 Términos de intercambio
Las operaciones entre columnas se hacen fila por fila, automáticamente.

In [ ]:
tabla["TI"] = tabla["IPX"] / tabla["IPM"] * 100

### 3.2 Crecimiento anual
`.pct_change()` calcula la variación respecto del año anterior. Multiplicamos por 100 para tenerla en %.

El primer año (2004) queda vacío porque no tiene año anterior.

In [ ]:
tabla["g_PBI"] = tabla["PBI"].pct_change() * 100
tabla["g_Gasto"] = tabla["Gasto_pub"].pct_change() * 100
tabla["g_Inversion"] = tabla["Inversion"].pct_change() * 100
tabla["g_TI"] = tabla["TI"].pct_change() * 100

### 3.3 Ratios respecto del PBI

In [ ]:
tabla["inv_pbi"] = tabla["Inversion"] / tabla["PBI"] * 100
tabla["gasto_pbi"] = tabla["Gasto_pub"] / tabla["PBI"] * 100
tabla["brecha"] = (tabla["Ahorro"] - tabla["Inversion"]) / tabla["PBI"] * 100

### 3.4 Rezago
`.shift(1)` baja la columna una fila: en 2010 queda el crecimiento de 2009.

In [ ]:
tabla["g_PBI_anterior"] = tabla["g_PBI"].shift(1)

### 3.5 Dummy de quiebre
Vale **1** en los años afectados por el cambio de escala del PBI y **0** en el resto.

Incluimos 2021 porque el cambio ocurrió a mitad de 2020, y eso también altera el crecimiento de 2021.

`.isin([...])` pregunta si el año está en la lista; `.astype(int)` convierte Sí/No en 1/0.

In [ ]:
tabla["quiebre"] = tabla["Anio"].isin([2006, 2020, 2021]).astype(int)

### 3.6 La tabla final

In [ ]:
tabla.round(2)

---
## 4. Gráficos

In [ ]:
plt.figure(figsize=(10, 3.5))
plt.bar(tabla["Anio"], tabla["g_PBI"], color="darkred")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Crecimiento anual del PBI (%)")
plt.show()

Las caídas de 2006, 2020 y 2021 son el problema de escala, por eso los marcamos con `quiebre`.

In [ ]:
plt.figure(figsize=(10, 3.5))
plt.plot(tabla["Anio"], tabla["TI"], marker="o", color="green")
plt.axhline(100, color="gray", linestyle="--")
plt.title("Términos de intercambio (IPX / IPM × 100)")
plt.show()

Por debajo de 100, los precios de exportación están relativamente peores que los de importación.

In [ ]:
plt.figure(figsize=(10, 3.5))
plt.plot(tabla["Anio"], tabla["inv_pbi"], marker="o", label="Inversión / PBI")
plt.plot(tabla["Anio"], tabla["gasto_pbi"], marker="o", label="Gasto / PBI")
plt.title("Ratios respecto del PBI (%)")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 3.5))
plt.bar(tabla["Anio"], tabla["brecha"], color="orange")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Brecha ahorro - inversión (% del PBI)")
plt.show()

La brecha es negativa todos los años: se invierte más de lo que se ahorra, y la diferencia se financia con ahorro externo.

---
## 5. Guardar
El archivo aparece en el panel **📁 Archivos**; clic derecho → **Descargar**.

In [ ]:
tabla.to_excel("tabla_anual.xlsx", index=False)
print("Guardado: tabla_anual.xlsx")

## Resumen

| Paso | Qué hicimos |
|---|---|
| Tipos | `"N.A"` → vacío con `pd.to_numeric` |
| Duplicados | valores copiados → vacío con `.duplicated()` |
| Errores | gasto = 0 y gasto diez veces menor → vacío |
| Imputar | `.interpolate()`: recta entre el trimestre anterior y el siguiente |
| Anual | `groupby("Anio")`: flujos se suman, índices se promedian |
| Unir | `merge(on="Anio")` → una fila por año |
| Variables | TI, crecimientos, ratios, brecha, rezago, dummy de quiebre |